# Phase 9 — PySpark Application Architecture Experiments Notebook

This is the **STARTER notebook** for Phase 9.

Run it **top-to-bottom**. The deterministic setup, experiment order,
architecture questions, and applied-project structure are preserved from the
SOLUTION notebook.

Worked implementation and answer-revealing conclusions have been removed so
you can design the boundaries yourself.

Use this workflow:

```text
identify responsibility
    ↓
identify coupling
    ↓
state the contract
    ↓
refactor ONE boundary
    ↓
run the same business logic
    ↓
verify correctness
    ↓
review maintainability
```

Core question:

> **Can another engineer understand, configure, test, run, change, and troubleshoot this pipeline without reverse-engineering one giant script?**

Important:

- Use **PySpark 4.2.0**.
- Use `from pyspark.sql import functions as F`.
- Use single-quoted Python strings.
- Keep business logic independent of paths, credentials, infrastructure, and orchestration.
- Preserve deterministic behavior.
- State grain before and after joins/aggregations.
- This notebook does **not** perform the formal mastery gate, update `ROADMAP.md`, mark Phase 9 complete, or enter Phase 10.


<a id="toc"></a>
## Table of Contents

- [Setup and Practice Data](#setup-and-practice-data)
- [Architecture Exercise Protocol](#architecture-exercise-protocol)
- [Experiment 1 — Decompose a Monolithic Pipeline](#experiment-1)
- [Experiment 2 — Reusable Transformation Functions](#experiment-2)
- [Experiment 3 — Parameterized Business Logic](#experiment-3)
- [Experiment 4 — Environment-Specific Configuration](#experiment-4)
- [Experiment 5 — Reader and Writer Boundaries](#experiment-5)
- [Experiment 6 — Thin Orchestration](#experiment-6)
- [Experiment 7 — Useful Logging](#experiment-7)
- [Experiment 8 — Exception Handling](#experiment-8)
- [Experiment 9 — Deterministic Behavior](#experiment-9)
- [Experiment 10 — Testable Transformation Design](#experiment-10)
- [Experiment 11 — Dependency Direction and Coupling](#experiment-11)
- [Experiment 12 — Dependency Management and Packaging](#experiment-12)
- [Applied Phase 9 Project](#applied-project)
- [Cleanup](#cleanup)


<a id="setup-and-practice-data"></a>
# Setup and Practice Data

Main grains:

```text
orders_df
= one row per order_id

customers_df
= one row per customer_id

customer_history_df
= one row per customer_record_id
```

The applied pipeline deliberately changes grain:

```text
order grain
    ↓
completed/qualified order grain
    ↓
enriched order grain
    ↓
province grain
```

[Back to Table of Contents](#toc)


In [ ]:
from dataclasses import dataclass
from datetime import date
from decimal import Decimal
import logging
from pathlib import Path
from tempfile import TemporaryDirectory

from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
from pyspark.sql import Window
from pyspark.sql import functions as F
from pyspark.sql.types import DateType
from pyspark.sql.types import DecimalType
from pyspark.sql.types import StringType
from pyspark.sql.types import StructField
from pyspark.sql.types import StructType


spark = (
    SparkSession.builder
    .appName('phase_09_application_architecture_experiments')
    .master('local[4]')
    # Keep the teaching workload small and predictable.
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')


In [ ]:
ORDERS_SCHEMA = StructType(
    [
        StructField('order_id', StringType(), nullable=False),
        StructField('customer_id', StringType(), nullable=False),
        StructField('order_date', DateType(), nullable=False),
        StructField('order_status', StringType(), nullable=False),
        StructField('net_sales', DecimalType(12, 2), nullable=False),
    ]
)

CUSTOMERS_SCHEMA = StructType(
    [
        StructField('customer_id', StringType(), nullable=False),
        StructField('customer_name', StringType(), nullable=False),
        StructField('province', StringType(), nullable=False),
        StructField('customer_segment', StringType(), nullable=False),
    ]
)

CUSTOMER_HISTORY_SCHEMA = StructType(
    [
        StructField('customer_record_id', StringType(), nullable=False),
        StructField('customer_id', StringType(), nullable=False),
        StructField('effective_date', DateType(), nullable=False),
        StructField('province', StringType(), nullable=False),
    ]
)

ORDERS_ROWS = [
    ('O001', 'C001', date(2026, 9, 1), 'COMPLETED', Decimal('125.00')),
    ('O002', 'C002', date(2026, 9, 1), 'CANCELLED', Decimal('80.00')),
    ('O003', 'C001', date(2026, 9, 2), 'COMPLETED', Decimal('45.00')),
    ('O004', 'C003', date(2026, 9, 2), 'COMPLETED', Decimal('200.00')),
    ('O005', 'C004', date(2026, 9, 3), 'COMPLETED', Decimal('30.00')),
]

CUSTOMERS_ROWS = [
    ('C001', 'Alice Wong', 'ON', 'CONSUMER'),
    ('C002', 'Ben Tremblay', 'QC', 'CONSUMER'),
    ('C003', 'Carla Singh', 'BC', 'BUSINESS'),
    ('C004', 'Diego Martin', 'ON', 'CONSUMER'),
]

CUSTOMER_HISTORY_ROWS = [
    ('R001', 'C001', date(2026, 1, 1), 'ON'),
    ('R002', 'C001', date(2026, 6, 1), 'QC'),
    ('R003', 'C002', date(2026, 5, 1), 'AB'),
    ('R004', 'C002', date(2026, 5, 1), 'QC'),
]

orders_df = spark.createDataFrame(ORDERS_ROWS, schema=ORDERS_SCHEMA)
customers_df = spark.createDataFrame(CUSTOMERS_ROWS, schema=CUSTOMERS_SCHEMA)
customer_history_df = spark.createDataFrame(
    CUSTOMER_HISTORY_ROWS,
    schema=CUSTOMER_HISTORY_SCHEMA,
)

orders_df.orderBy('order_id').show(truncate=False)
customers_df.orderBy('customer_id').show(truncate=False)


<a id="architecture-exercise-protocol"></a>
# Architecture Exercise Protocol

For each experiment, ask:

```text
1. What responsibility is this code performing?
2. What inputs does it genuinely need?
3. What should it deliberately NOT know?
4. Does it perform side effects?
5. Can the business logic be called directly with in-memory DataFrames?
6. Is environment/run state explicit?
7. Is behavior deterministic?
8. Can a failure be localized?
9. Would another engineer know where to change this behavior?
```

The target dependency direction is approximately:

```text
configuration
      ↓
orchestration
  ├── readers ─────→ schemas
  ├── validation
  ├── transformations
  └── writers
```

Business transformations should stay near the center because they are the
easiest layer to reuse and test when infrastructure concerns do not leak into
them.

[Back to Table of Contents](#toc)


<a id="experiment-1"></a>
# Experiment 1 — Decompose a Monolithic Pipeline

A monolithic script often mixes:

```text
path selection
reading
schema assumptions
validation
business filters
joins
aggregation
logging
writing
error handling
```

The first architecture task is not to create seven files immediately. It is to
identify **responsibilities and boundaries**.

[Back to Table of Contents](#toc)


In [ ]:
MONOLITHIC_ANTI_PATTERN = (
    'def build_sales_report(spark):\n'
    '    environment = \'prod\'\n'
    '    orders_path = \'/prod/orders\'\n'
    '    customers_path = \'/prod/customers\'\n'
    '    output_path = \'/prod/sales_by_province\'\n'
    '\n'
    '    orders_df = spark.read.parquet(orders_path)\n'
    '    customers_df = spark.read.parquet(customers_path)\n'
    '\n'
    '    result_df = (\n'
    '        orders_df\n'
    '        .filter(F.col(\'order_status\') == \'COMPLETED\')\n'
    '        .join(customers_df, on=\'customer_id\', how=\'left\')\n'
    '        .groupBy(\'province\')\n'
    '        .agg(F.sum(\'net_sales\').alias(\'net_sales\'))\n'
    '    )\n'
    '\n'
    '    result_df.write.mode(\'overwrite\').parquet(output_path)\n'
)

print(MONOLITHIC_ANTI_PATTERN)

# TODO: classify each concern into configuration, I/O, validation,
# transformation, writer, or orchestration.
responsibility_map = {}

responsibility_map


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-2"></a>
# Experiment 2 — Reusable Transformation Functions

The stable center of the application should resemble:

```python
def transform_orders(orders_df, customers_df):
    ...
    return result_df
```

A transformation should depend on the data and explicit business parameters,
not on filesystem layout or runtime infrastructure.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 2 — implement require_columns() and transform_orders().
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 2 — verify columns, row count, and order_id grain.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-3"></a>
# Experiment 3 — Parameterized Business Logic

A reusable function should accept legitimate run/business variation explicitly.

Bad hidden state:

```text
function silently reads a global status list
function checks machine environment variables mid-transformation
function calls today's date internally when replayability matters
```

Better:

```text
caller supplies the values as explicit parameters
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 3 — implement parameterized filter_orders().
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 3 — verify both parameterized outputs.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-4"></a>
# Experiment 4 — Environment-Specific Configuration

Environment-specific infrastructure values should change without changing
transformation code.

We use an immutable configuration object so one pipeline execution has a clear
set of settings.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 4 — implement PipelineConfig and build_config().
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 4 — compare dev/test/prod configuration.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-5"></a>
# Experiment 5 — Reader and Writer Boundaries

Readers translate external storage into DataFrames.

Writers translate DataFrames into external persistence.

Neither should redefine the business result.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 5 — implement reader and writer boundaries.
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 5 — seed, reread, and verify schemas.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-6"></a>
# Experiment 6 — Thin Orchestration

Orchestration inside the Spark application should read like the pipeline
workflow.

It coordinates responsibilities instead of containing the detailed business
logic itself.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 6 — implement the business transformation sequence.
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 6 — run the in-memory business result.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-7"></a>
# Experiment 7 — Useful Logging

Logging should describe application state and major boundaries without creating
unnecessary Spark jobs solely for log decoration.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 7 — configure useful application logging.
# Use the experiment requirements above and the Phase 9 guide.


Do **not** automatically do this merely for a log message:

```python
logger.info('Rows=%s', df.count())
```

`count()` is a real Spark action. Run it only when the count itself is an
intentional validation/metric requirement.

A useful log normally captures:

```text
pipeline start/finish
environment/run parameters
major processing boundary
input/output location
failure context
```

[Back to Table of Contents](#toc)


<a id="experiment-8"></a>
# Experiment 8 — Exception Handling

Good exception handling preserves failure information.

Catch only when you can:

```text
recover
add useful context
translate to a meaningful application-level error
```

Do not catch every exception just to print a message and continue.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 8 — implement exception handling with preserved causes.
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 8 — trigger and inspect a deliberate failure.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-9"></a>
# Experiment 9 — Deterministic Behavior

Determinism means:

```text
same logical inputs + same explicit parameters
→ same logical business output
```

This requires attention to tie-breaking, run-dependent values, randomness, and
implicit assumptions about row order.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 9 — implement deterministic latest-record selection.
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Experiment 9 — verify deterministic record choice and run date.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Explain what makes latest-record selection deterministic.

Identify any run-dependent values that should be injected rather than
discovered implicitly.

[Back to Table of Contents](#toc)


<a id="experiment-10"></a>
# Experiment 10 — Testable Transformation Design

A well-designed transformation can be tested with tiny in-memory DataFrames.

The test does not need:

```text
production paths
credentials
cloud services
scheduler
real output table
```

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 10 — write in-memory business correctness checks.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-11"></a>
# Experiment 11 — Dependency Direction and Coupling

A maintainable dependency graph should keep business logic from depending
outward on infrastructure.

Preferred direction:

```text
pipeline/orchestration
  ├── config
  ├── readers ─────→ schemas
  ├── validation
  ├── transformations
  └── writers
```

Transformations should not import the pipeline entry point, inspect deployment
environment state, or decide storage destinations.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 11 — define dependency contracts for each layer.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Record what this experiment demonstrates about responsibility boundaries,
coupling, determinism, testability, or maintainability.

[Back to Table of Contents](#toc)


<a id="experiment-12"></a>
# Experiment 12 — Dependency Management and Packaging

Dependency management answers:

```text
What external software does this application require?
```

Packaging answers:

```text
How is the application code organized as an importable/deployable unit?
```

These concerns are related but not identical.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Experiment 12 — propose dependency and package structure.
# Use the experiment requirements above and the Phase 9 guide.


### Your analysis

Explain the difference between dependency management and packaging.

Then justify the package structure you proposed without adding modules that
exist only to forward another function call.

[Back to Table of Contents](#toc)


<a id="applied-project"></a>
# Applied Phase 9 Project

## Refactor a Retail Pipeline Into an Application

This integrated run combines the boundaries from the previous experiments.

Target workflow:

```text
configuration
    ↓
read orders/customers
    ↓
validate required keys
    ↓
filter orders
    ↓
enrich orders
    ↓
aggregate to province
    ↓
add deterministic run date
    ↓
write output
```

Expected output grain:

```text
one row per province
```

The orchestration function should be understandable without reading the
implementation of every transformation.

[Back to Table of Contents](#toc)


In [ ]:
# TODO: Applied Project — implement validation and run_pipeline().
# Use the experiment requirements above and the Phase 9 guide.


In [ ]:
# TODO: Applied Project — run and reconcile the complete pipeline.
# Use the experiment requirements above and the Phase 9 guide.


## Applied Architecture Review

After completing the project, answer:

```text
Where is the completed-order rule?
Where are environment paths defined?
Which functions are directly testable with in-memory DataFrames?
Which functions intentionally perform I/O?
Where is logging configured?
Where is pipeline-level failure context added?
How is run_date deterministic?
What dependencies are declared externally?
What changes if Parquet output becomes a warehouse write?
What should remain unchanged?
```

Success means someone unfamiliar with the original author can locate the
business rules, schemas, I/O boundaries, configuration, orchestration,
side effects, test seams, failure boundaries, deterministic inputs, and
dependency/package expectations.

This notebook is practice only. It is not the formal mastery gate.

[Back to Table of Contents](#toc)


<a id="cleanup"></a>
# Cleanup

Stop the local Spark application after completing the notebook.

[Back to Table of Contents](#toc)


In [ ]:
spark.stop()

print('Phase 9 solution notebook complete.')
